[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/09_ONNX_Graph_Manipulation/04_Model_Validation/Model_Validation_Deep_Dive.ipynb)

# 9.4 Model Validation — Deep Dive

Model validation ensures that an ONNX model is structurally correct, semantically well-formed,
and numerically faithful to its reference implementation. This notebook covers the formal
well-formedness conditions, the `onnx.checker` module, common errors and repair strategies,
custom policy validation, and conformance testing.

## Table of Contents

| # | Section | Topic |
|---|---------|-------|
| 1 | [Checker Rules and Well-Formedness](#section-1) | Formal conditions for valid ONNX graphs |
| 2 | [The onnx.checker Module](#section-2) | check_model, check_graph, check_node |
| 3 | [Common Validation Errors](#section-3) | Shape mismatches, missing initializers, invalid attributes |
| 4 | [Validation Pipeline](#section-4) | Load → check → infer → runtime test |
| 5 | [Custom Policy Validation](#section-5) | Forbidden ops, opset bounds, size budgets |
| 6 | [Conformance Testing](#section-6) | Backend tests, numerical accuracy, cross-runtime checks |
| 7 | [Repair Strategies](#section-7) | Fixing errors, opset conversion, adding initializers |

<a id='section-1'></a>
## Section 1: Checker Rules and Well-Formedness

### Formal Well-Formedness Predicate

An ONNX graph $G$ is **well-formed** if and only if it satisfies five conditions simultaneously:

$$\text{WF}(G) \iff \text{C1}(G) \wedge \text{C2}(G) \wedge \text{C3}(G) \wedge \text{C4}(G) \wedge \text{C5}(G)$$

Each condition is a necessary property of a valid graph. Violating any single condition
makes the entire graph invalid.

### Condition 1: Defined-Before-Use (Input Availability)

Every input to every node must be **defined** before it is consumed — by a graph input,
an initializer, or the output of a topologically preceding node:

$$\forall n \in \text{nodes}(G),\; \forall i \in \text{inputs}(n):\; i \in \text{defined\_before}(n)$$

where $\text{defined\_before}(n) = \text{graph\_inputs}(G) \cup \text{initializers}(G) \cup \bigcup_{m \prec n} \text{outputs}(m)$.

### Condition 2: Name Uniqueness (No Conflicts)

Every value name in the graph must be unique — no two distinct value producers may
share the same name:

$$\forall v_1, v_2 \in \text{values}(G):\; v_1 \neq v_2 \implies \text{name}(v_1) \neq \text{name}(v_2)$$

This is the **single static assignment** (SSA) property.

### Condition 3: Op Schema Compliance

Each node must conform to the operator schema defined for its `op_type` at the
declared opset version — correct number of inputs, valid attribute names and types:

$$\forall n \in \text{nodes}(G):\; \text{schema}(\text{op}(n), \text{opset}(G)) \vdash n$$

### Condition 4: Type Consistency

Declared types must match inferred types. If a value $v$ has a declared type annotation
and the type inference engine infers a different type, the model is invalid:

$$\text{type}(v) \in \text{allowed\_types}(\text{op}(v), \text{position}(v))$$

### Condition 5: Valid DAG (No Cycles)

The computation graph must be a directed acyclic graph. Cycles would create
infinite computation loops:

$$\nexists\; n_1, n_2, \ldots, n_k \in \text{nodes}(G):\; n_1 \to n_2 \to \cdots \to n_k \to n_1$$

### Well-Formedness Checker Flow

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    WELL-FORMEDNESS CHECKER FLOW                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ModelProto                                                                 │
│      │                                                                      │
│      ▼                                                                      │
│  ┌─────────────────────┐    FAIL                                            │
│  │ C1: Defined-Before  │──────────▶ "Unresolved reference: <name>"          │
│  │     -Use Check      │                                                    │
│  └─────────┬───────────┘                                                    │
│            │ PASS                                                           │
│            ▼                                                                │
│  ┌─────────────────────┐    FAIL                                            │
│  │ C2: Name Uniqueness │──────────▶ "Duplicate name: <name>"                │
│  │     (SSA) Check     │                                                    │
│  └─────────┬───────────┘                                                    │
│            │ PASS                                                           │
│            ▼                                                                │
│  ┌─────────────────────┐    FAIL                                            │
│  │ C3: Op Schema       │──────────▶ "No schema for <op> in opset <ver>"     │
│  │     Compliance      │                                                    │
│  └─────────┬───────────┘                                                    │
│            │ PASS                                                           │
│            ▼                                                                │
│  ┌─────────────────────┐    FAIL                                            │
│  │ C4: Type            │──────────▶ "Type mismatch on <value>"              │
│  │     Consistency     │                                                    │
│  └─────────┬───────────┘                                                    │
│            │ PASS                                                           │
│            ▼                                                                │
│  ┌─────────────────────┐    FAIL                                            │
│  │ C5: DAG Validation  │──────────▶ "Cycle detected involving <node>"       │
│  │     (No Cycles)     │                                                    │
│  └─────────┬───────────┘                                                    │
│            │ PASS                                                           │
│            ▼                                                                │
│       ✓ WELL-FORMED                                                        │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
!pip install onnx onnxruntime numpy matplotlib -q

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper, checker, shape_inference
from onnx.checker import ValidationError
import onnxruntime as ort
import traceback

print(f"ONNX version:        {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")
print(f"NumPy version:       {np.__version__}")

In [ ]:
# --- Demonstrate all 5 well-formedness conditions ---

def check_and_report(model, label):
    """Run check_model and print a concise result."""
    try:
        checker.check_model(model)
        print(f"  [{label}]  PASS")
        return True
    except ValidationError as e:
        msg = str(e).split('\n')[0][:120]
        print(f"  [{label}]  FAIL  →  {msg}")
        return False
    except Exception as e:
        msg = str(e).split('\n')[0][:120]
        print(f"  [{label}]  ERROR →  {msg}")
        return False


# ── C1: Defined-Before-Use ──
print("Condition 1: All inputs must be defined")
print("=" * 50)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
bad_node = helper.make_node("Add", ["X", "UNDEFINED"], ["Y"])
graph = helper.make_graph([bad_node], "c1_bad", [X], [Y])
model_c1_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_c1_bad, "C1-violate: undefined input 'UNDEFINED'")

B_np = np.zeros(10, dtype=np.float32)
B_init = numpy_helper.from_array(B_np, name="B")
good_node = helper.make_node("Add", ["X", "B"], ["Y"])
graph = helper.make_graph([good_node], "c1_good", [X], [Y], initializer=[B_init])
model_c1_good = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_c1_good, "C1-fixed: 'B' provided as initializer")


# ── C2: Name Uniqueness (SSA) ──
print("\nCondition 2: No duplicate value names")
print("=" * 50)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
n1 = helper.make_node("Relu", ["X"], ["T"])
n2 = helper.make_node("Relu", ["T"], ["T"])  # reuses 'T' as output
n3 = helper.make_node("Relu", ["T"], ["Y"])
graph = helper.make_graph([n1, n2, n3], "c2_bad", [X], [Y])
model_c2_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_c2_bad, "C2-violate: 'T' assigned twice")


# ── C3: Op Schema Compliance ──
print("\nCondition 3: Operator schema compliance")
print("=" * 50)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
bad_op = helper.make_node("NonExistentOp", ["X"], ["Y"])
graph = helper.make_graph([bad_op], "c3_bad", [X], [Y])
model_c3_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_c3_bad, "C3-violate: unknown op 'NonExistentOp'")


# ── C4: Type Consistency ──
print("\nCondition 4: Type consistency")
print("=" * 50)

X_f = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
W_i = numpy_helper.from_array(np.zeros((10, 5), dtype=np.int32), name="W")
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
mm = helper.make_node("MatMul", ["X", "W"], ["Y"])
graph = helper.make_graph([mm], "c4_bad", [X_f], [Y], initializer=[W_i])
model_c4_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_c4_bad, "C4-violate: float32 MatMul int32")


# ── C5: Valid DAG ──
print("\nCondition 5: Graph must be a DAG (no cycles)")
print("=" * 50)
print("  [C5-note] The checker enforces topological ordering via C1.")
print("  A cycle n1→n2→n1 means n1's input is n2's output, but n2")
print("  comes after n1, so n2's output is not defined before n1.")
print("  Hence C1 implicitly enforces acyclicity.")

<a id='section-2'></a>
## Section 2: The `onnx.checker` Module

The `onnx.checker` module provides three levels of validation:

| Function | Scope | Input |
|----------|-------|-------|
| `check_model(model)` | Full model (opset, graph, metadata) | `ModelProto` |
| `check_graph(graph)` | Graph structure only | `GraphProto` |
| `check_node(node, ctx)` | Single node against its schema | `NodeProto` |

All three raise `onnx.checker.ValidationError` on failure.

```
                        onnx.checker Module
                    ┌──────────────────────────┐
                    │                          │
                    │   check_model(model)     │ ← Full model: IR version,
                    │       │                  │   opset imports, graph,
                    │       │ delegates to     │   producer metadata
                    │       ▼                  │
                    │   check_graph(graph)     │ ← Graph: inputs, outputs,
                    │       │                  │   initializers, node wiring
                    │       │ iterates over    │
                    │       ▼                  │
                    │   check_node(node, ctx)  │ ← Node: op schema match,
                    │                          │   input count, attributes
                    └──────────────────────────┘
                              │
                    On failure: raise ValidationError
                    On success: return None (silent)
```

In [ ]:
# --- check_model: Full model validation ---

np.random.seed(42)
W_np = np.random.randn(784, 10).astype(np.float32)
B_np = np.random.randn(10).astype(np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["N", 10])
W = numpy_helper.from_array(W_np, name="W")
B = numpy_helper.from_array(B_np, name="B")

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["X", "W"], ["XW"]),
        helper.make_node("Add", ["XW", "B"], ["Y"]),
    ],
    "linear_model", [X], [Y], initializer=[W, B],
)
valid_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print("=== check_model demo ===")
try:
    checker.check_model(valid_model)
    print("check_model: PASSED (no exception raised)")
except ValidationError as e:
    print(f"check_model: FAILED — {e}")

print(f"\nModel details:")
print(f"  IR version:  {valid_model.ir_version}")
print(f"  Opset:       {valid_model.opset_import[0].version}")
print(f"  Nodes:       {len(valid_model.graph.node)}")
print(f"  Inputs:      {[i.name for i in valid_model.graph.input]}")
print(f"  Outputs:     {[o.name for o in valid_model.graph.output]}")
print(f"  Initializers: {[i.name for i in valid_model.graph.initializer]}")

In [ ]:
# --- check_graph and check_node ---

print("=== check_graph demo ===")
try:
    checker.check_graph(valid_model.graph)
    print("check_graph: PASSED")
except ValidationError as e:
    print(f"check_graph: FAILED — {e}")

print("\n=== check_node demo ===")
relu_node = helper.make_node("Relu", ["input"], ["output"])
ctx = checker.C.CheckerContext()
ctx.ir_version = onnx.IR_VERSION
ctx.opset_imports = {"" : 17}
try:
    checker.check_node(relu_node, ctx)
    print("check_node(Relu): PASSED")
except ValidationError as e:
    print(f"check_node(Relu): FAILED — {e}")

print("\n=== ValidationError exception details ===")
bad_node = helper.make_node("FakeOp", ["x"], ["y"])
try:
    checker.check_node(bad_node, ctx)
except ValidationError as e:
    print(f"Exception type: {type(e).__name__}")
    print(f"Exception module: {type(e).__module__}")
    print(f"Message (first 200 chars): {str(e)[:200]}")

<a id='section-3'></a>
## Section 3: Common Validation Errors

### Error Taxonomy

```
                        ONNX Validation Errors
                    ┌──────────────┴──────────────┐
                    │                             │
              Structural                    Semantic
           ┌──────┼──────┐           ┌──────┼──────┐
           │      │      │           │      │      │
      Undefined  Name   Op       Shape   Type    Opset
       Input   Clash  Schema  Mismatch  Error  Version
           │      │      │           │      │      │
           ▼      ▼      ▼           ▼      ▼      ▼
       "input   "dup   "no        "dim   "float  "op not
        'Z' not  name   schema     error   vs     in opset
        defined" 'T'"   for op"    on     int"    13"
                                   Add"
```

### Common Error Patterns

| Error Category | Typical Cause | Checker Stage |
|:---|:---|:---|
| Undefined input | Typo in tensor name or missing initializer | C1 (defined-before-use) |
| Shape mismatch | Binary op with incompatible dimensions | Shape inference |
| Missing initializer | Name mismatch between node input and initializer | C1 |
| Invalid attribute | Wrong type (int vs float) or out of range | C3 (schema) |
| Unsupported opset | Op used from a newer opset than declared | C3 (schema) |
| External data issues | Missing `.onnx_data` file or wrong path | Load-time |

In [ ]:
# --- Error 1: Shape mismatch (binary op dimension error) ---

print("Error 1: Shape Mismatch")
print("=" * 60)

A = helper.make_tensor_value_info("A", TensorProto.FLOAT, [4, 3])
B = helper.make_tensor_value_info("B", TensorProto.FLOAT, [5, 3])
C_out = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)

matmul = helper.make_node("MatMul", ["A", "B"], ["C"])
graph = helper.make_graph([matmul], "shape_mismatch", [A, B], [C_out])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

check_and_report(model, "Structural check (passes — checker doesn't verify shapes)")

try:
    inferred = shape_inference.infer_shapes(model, check_type=True)
    out_shape = inferred.graph.output[0].type.tensor_type.shape
    dims = [d.dim_value for d in out_shape.dim]
    print(f"  Shape inference result: C = {dims}")
    print("  Note: MatMul([4,3], [5,3]) is invalid (inner dims 3≠5).")
    print("  Shape inference may not always catch this — runtime will.")
except Exception as e:
    print(f"  Shape inference caught error: {str(e)[:150]}")

try:
    sess = ort.InferenceSession(model.SerializeToString())
    a_np = np.random.randn(4, 3).astype(np.float32)
    b_np = np.random.randn(5, 3).astype(np.float32)
    sess.run(None, {"A": a_np, "B": b_np})
except Exception as e:
    print(f"  Runtime caught: {str(e).split(chr(10))[0][:120]}")

In [ ]:
# --- Error 2: Missing initializer (name mismatch) ---

print("Error 2: Missing Initializer")
print("=" * 60)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

W_init = numpy_helper.from_array(np.zeros((10, 5), dtype=np.float32), name="weights")
mm = helper.make_node("MatMul", ["X", "W"], ["Y"])  # refers to 'W' not 'weights'

graph = helper.make_graph([mm], "missing_init", [X], [Y], initializer=[W_init])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model, "Node refers to 'W' but initializer is named 'weights'")

print("\n  Fix: rename the initializer to match the node input")
W_init_fixed = numpy_helper.from_array(np.zeros((10, 5), dtype=np.float32), name="W")
graph_fixed = helper.make_graph([mm], "fixed_init", [X], [Y], initializer=[W_init_fixed])
model_fixed = helper.make_model(graph_fixed, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_fixed, "Fixed: initializer renamed to 'W'")


# --- Error 3: Invalid attribute ---

print("\nError 3: Invalid Attribute")
print("=" * 60)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 8, 8])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W_conv = numpy_helper.from_array(
    np.random.randn(16, 3, 3, 3).astype(np.float32), name="W"
)

conv_bad = helper.make_node(
    "Conv", ["X", "W"], ["Y"],
    kernel_shape=[3, 3],
    pads=[1, 1, 1, 1],
    strides=[0, 0],  # stride of 0 is invalid
)
graph = helper.make_graph([conv_bad], "bad_attr", [X], [Y], initializer=[W_conv])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model, "Conv with stride=[0,0]")
print("  Note: checker may not catch semantic invalidity; runtime will fail.")


# --- Error 4: Unsupported opset version ---

print("\nError 4: Unsupported Opset Version")
print("=" * 60)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 768])
gamma = numpy_helper.from_array(np.ones(768, dtype=np.float32), name="gamma")
beta = numpy_helper.from_array(np.zeros(768, dtype=np.float32), name="beta")
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

ln = helper.make_node("LayerNormalization", ["X", "gamma", "beta"], ["Y"],
                      epsilon=1e-5, axis=-1)
graph = helper.make_graph([ln], "old_opset", [X], [Y], initializer=[gamma, beta])
model_old = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
check_and_report(model_old, "LayerNorm in opset 13 (introduced in 17)")

model_new = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
check_and_report(model_new, "Fixed: LayerNorm with opset 17")

<a id='section-4'></a>
## Section 4: Validation Pipeline

A production validation pipeline chains multiple checks in order of increasing cost:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    VALIDATION PIPELINE                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌──────────────┐   ~1ms    ┌──────────────┐   ~10ms   ┌──────────────┐    │
│  │   Load       │──────────▶│ check_model  │──────────▶│shape_infer   │    │
│  │   Model      │           │ (structural) │           │(type check)  │    │
│  └──────┬───────┘           └──────┬───────┘           └──────┬───────┘    │
│         │ FAIL:                    │ FAIL:                    │ FAIL:       │
│         │ corrupt                  │ malformed                │ shape/type  │
│         │ protobuf                 │ graph                   │ error       │
│         ▼                          ▼                         ▼             │
│      REJECT                     REJECT                   REJECT           │
│                                                              │ PASS        │
│                                                              ▼             │
│                                                    ┌──────────────┐        │
│                                                    │ Runtime Test │        │
│                                                    │ (ORT session │        │
│                                                    │  N samples)  │        │
│                                                    └──────┬───────┘        │
│                                                           │ ~100ms         │
│                                                           ▼               │
│                                                     ACCEPT / REJECT       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### CI/CD Integration Pattern

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    CI/CD VALIDATION GATE                                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  git push / PR                                                              │
│      │                                                                      │
│      ▼                                                                      │
│  ┌─────────────┐     ┌──────────────┐     ┌──────────────┐                 │
│  │ Export Model │────▶│ Validate     │────▶│ Policy Check │                 │
│  │ (training   │     │ (structural  │     │ (forbidden   │                 │
│  │  pipeline)  │     │  + numeric)  │     │  ops, size)  │                 │
│  └─────────────┘     └──────┬───────┘     └──────┬───────┘                 │
│                             │                     │                         │
│                     FAIL: block merge      FAIL: block merge               │
│                     PASS: continue ──────▶ PASS: approve PR                │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# --- Full validation pipeline implementation ---

def validate_pipeline(model_proto, test_inputs=None, reference_fn=None,
                      atol=1e-5, rtol=1e-4):
    """
    Production validation pipeline:
      Stage 1: Structural check (check_model)
      Stage 2: Shape/type inference
      Stage 3: Runtime creation test
      Stage 4: Numerical accuracy (if reference provided)

    Returns dict with per-stage pass/fail and diagnostics.
    """
    report = {
        "stage1_structural": None,
        "stage2_shape_inference": None,
        "stage3_runtime": None,
        "stage4_numerical": None,
        "errors": [],
    }

    # Stage 1
    try:
        checker.check_model(model_proto)
        report["stage1_structural"] = "PASS"
    except Exception as e:
        report["stage1_structural"] = "FAIL"
        report["errors"].append(f"Stage 1: {str(e)[:200]}")
        return report

    # Stage 2
    try:
        inferred = shape_inference.infer_shapes(model_proto, check_type=True)
        report["stage2_shape_inference"] = "PASS"
        shapes = {}
        for vi in list(inferred.graph.value_info) + list(inferred.graph.output):
            tt = vi.type.tensor_type
            if tt.HasField("shape"):
                dims = [d.dim_value if d.dim_value > 0 else d.dim_param
                        for d in tt.shape.dim]
                shapes[vi.name] = dims
        report["inferred_shapes"] = shapes
    except Exception as e:
        report["stage2_shape_inference"] = "FAIL"
        report["errors"].append(f"Stage 2: {str(e)[:200]}")
        return report

    # Stage 3
    try:
        sess = ort.InferenceSession(model_proto.SerializeToString())
        report["stage3_runtime"] = "PASS"
        report["runtime_inputs"] = [
            {"name": i.name, "shape": i.shape, "type": i.type}
            for i in sess.get_inputs()
        ]
    except Exception as e:
        report["stage3_runtime"] = "FAIL"
        report["errors"].append(f"Stage 3: {str(e)[:200]}")
        return report

    # Stage 4
    if test_inputs is not None and reference_fn is not None:
        input_name = sess.get_inputs()[0].name
        max_diffs = []
        for x_np in test_inputs:
            y_ref = reference_fn(x_np)
            y_ort = sess.run(None, {input_name: x_np})[0]
            max_diffs.append(np.max(np.abs(y_ref - y_ort)))
        all_pass = all(d < atol + rtol * np.max(np.abs(y_ref)) for d in max_diffs)
        report["stage4_numerical"] = "PASS" if all_pass else "FAIL"
        report["max_diff"] = max(max_diffs)
        report["mean_diff"] = np.mean(max_diffs)
    else:
        report["stage4_numerical"] = "SKIP (no reference)"

    return report


# Run the pipeline on a valid model
def ref_fn(x):
    return np.maximum(0, x @ W_np + B_np)

W_np = np.random.randn(784, 10).astype(np.float32)
B_np = np.random.randn(10).astype(np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W = numpy_helper.from_array(W_np, name="W")
B = numpy_helper.from_array(B_np, name="B")

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["X", "W"], ["XW"]),
        helper.make_node("Add", ["XW", "B"], ["H"]),
        helper.make_node("Relu", ["H"], ["Y"]),
    ],
    "pipeline_demo", [X], [Y], initializer=[W, B],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

test_data = [np.random.randn(8, 784).astype(np.float32) for _ in range(20)]
report = validate_pipeline(model, test_data, ref_fn)

print("Validation Pipeline Report")
print("=" * 50)
print(f"  Stage 1 (structural):       {report['stage1_structural']}")
print(f"  Stage 2 (shape inference):  {report['stage2_shape_inference']}")
print(f"  Stage 3 (runtime creation): {report['stage3_runtime']}")
print(f"  Stage 4 (numerical):        {report['stage4_numerical']}")
if 'max_diff' in report:
    print(f"  Max numerical diff:         {report['max_diff']:.2e}")
if report['errors']:
    print(f"  Errors: {report['errors']}")

In [ ]:
# --- CI/CD integration: exit-code-based validation script ---

import sys
import json

def ci_validate(model_bytes, test_inputs=None, reference_fn=None,
                atol=1e-5, rtol=1e-4):
    """
    CI-grade validation function.
    Returns exit code: 0 = success, 1 = failure.
    Prints JSON report to stdout for CI log parsing.
    """
    model = onnx.load_from_string(model_bytes)
    report = validate_pipeline(model, test_inputs, reference_fn, atol, rtol)

    failed_stages = [k for k, v in report.items()
                     if isinstance(v, str) and v == "FAIL"]

    result = {
        "status": "FAIL" if failed_stages else "PASS",
        "failed_stages": failed_stages,
        "errors": report.get("errors", []),
        "max_diff": report.get("max_diff"),
    }
    return result


result = ci_validate(
    model.SerializeToString(),
    test_data, ref_fn,
)

print("CI Validation Result (JSON):")
print(json.dumps(result, indent=2, default=str))

print(f"\nCI exit code would be: {0 if result['status'] == 'PASS' else 1}")
print("\nExample CI script usage:")
print("  python validate_model.py model.onnx --atol 1e-5 --rtol 1e-4")
print("  # exits 0 on success, 1 on failure")

<a id='section-5'></a>
## Section 5: Custom Policy Validation

Beyond structural correctness, organizations enforce **policies** — constraints that
go beyond the ONNX spec:

| Policy | Rationale |
|:---|:---|
| Forbidden op detection | Some ops are not supported by target hardware |
| Opset version bounds | Ensure compatibility with deployed runtimes |
| Model size budget | Edge devices have storage limits |
| Quantization contract | INT8 models must follow specific patterns |
| Naming conventions | Team standards for node/tensor names |

These checks are implemented as Python functions that walk the graph and
raise `PolicyViolation` on non-compliance.

```
                     Custom Policy Layer
                ┌──────────────────────────┐
                │                          │
                │  ┌────────────────────┐  │
                │  │ Forbidden Ops      │  │  "No Resize, no Loop"
                │  └────────────────────┘  │
                │  ┌────────────────────┐  │
                │  │ Opset Bounds       │  │  "opset ∈ [13, 18]"
                │  └────────────────────┘  │
                │  ┌────────────────────┐  │
                │  │ Size Budget        │  │  "≤ 50 MB"
                │  └────────────────────┘  │
                │  ┌────────────────────┐  │
                │  │ Quant Contract     │  │  "DequantizeLinear after every Q"
                │  └────────────────────┘  │
                │  ┌────────────────────┐  │
                │  │ Naming Convention  │  │  "snake_case, prefixed"
                │  └────────────────────┘  │
                │                          │
                └──────────────────────────┘
```

In [ ]:
# --- Custom policy validation framework ---

import re


class PolicyViolation(Exception):
    """Raised when a model violates an organizational policy."""
    pass


def check_forbidden_ops(model, forbidden=None):
    """Reject models containing ops not supported by target hardware."""
    if forbidden is None:
        forbidden = {"Loop", "If", "Scan", "Resize", "NonMaxSuppression"}
    violations = []
    for node in model.graph.node:
        if node.op_type in forbidden:
            violations.append(
                f"Forbidden op '{node.op_type}' at node "
                f"'{node.name or '(unnamed)'}' "
                f"(outputs: {list(node.output)})"
            )
    return violations


def check_opset_bounds(model, min_opset=13, max_opset=18):
    """Ensure the model's opset version is within deployment bounds."""
    violations = []
    for imp in model.opset_import:
        domain = imp.domain or "ai.onnx"
        ver = imp.version
        if domain in ("", "ai.onnx"):
            if ver < min_opset:
                violations.append(
                    f"Opset {ver} < minimum {min_opset} for domain '{domain}'"
                )
            if ver > max_opset:
                violations.append(
                    f"Opset {ver} > maximum {max_opset} for domain '{domain}'"
                )
    return violations


def check_model_size(model, max_bytes=50 * 1024 * 1024):
    """Enforce model size budget (default: 50 MB)."""
    size = len(model.SerializeToString())
    if size > max_bytes:
        return [f"Model size {size / 1e6:.1f} MB exceeds budget {max_bytes / 1e6:.1f} MB"]
    return []


def check_naming_convention(model, pattern=r'^[a-z][a-z0-9_]*$'):
    """Check that all node output names follow a naming convention."""
    violations = []
    regex = re.compile(pattern)
    for node in model.graph.node:
        for out in node.output:
            if out and not regex.match(out):
                violations.append(
                    f"Output '{out}' of {node.op_type} does not match "
                    f"pattern '{pattern}'"
                )
    return violations


def check_quantization_contract(model):
    """Verify that every QuantizeLinear is followed by DequantizeLinear."""
    q_outputs = set()
    dq_inputs = set()
    for node in model.graph.node:
        if node.op_type == "QuantizeLinear":
            q_outputs.update(node.output)
        if node.op_type == "DequantizeLinear":
            dq_inputs.update(node.input)
    orphan_q = q_outputs - dq_inputs
    if orphan_q:
        return [f"QuantizeLinear output(s) {orphan_q} not consumed by DequantizeLinear"]
    return []


def run_all_policies(model, policies=None):
    """Run all policy checks and aggregate violations."""
    if policies is None:
        policies = [
            ("Forbidden ops", lambda m: check_forbidden_ops(m)),
            ("Opset bounds", lambda m: check_opset_bounds(m)),
            ("Model size", lambda m: check_model_size(m)),
            ("Naming convention", lambda m: check_naming_convention(m)),
            ("Quantization contract", lambda m: check_quantization_contract(m)),
        ]
    all_violations = {}
    for name, check_fn in policies:
        v = check_fn(model)
        if v:
            all_violations[name] = v
    return all_violations


# Demonstrate policy checks
print("Policy Validation Demo")
print("=" * 60)

violations = run_all_policies(valid_model)
if violations:
    for policy, vlist in violations.items():
        print(f"\n  POLICY FAIL: {policy}")
        for v in vlist:
            print(f"    - {v}")
else:
    print("  All policies passed (no naming convention enforced on this model).")

print("\n--- Testing with policy violations ---")
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 8, 8])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
resize = helper.make_node("Resize", ["X", "", "", ""], ["Y"])
graph = helper.make_graph([resize], "policy_test", [X], [Y])
policy_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 11)])

violations = run_all_policies(policy_model)
for policy, vlist in violations.items():
    print(f"  POLICY FAIL: {policy}")
    for v in vlist[:2]:
        print(f"    - {v}")

In [ ]:
# --- Quantization contract enforcement demo ---

print("Quantization Contract Enforcement")
print("=" * 60)

X = helper.make_tensor_value_info("x", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("y", TensorProto.FLOAT, None)

scale = numpy_helper.from_array(np.array([0.1], dtype=np.float32), name="scale")
zero_pt = numpy_helper.from_array(np.array([128], dtype=np.uint8), name="zero_pt")

q_node = helper.make_node("QuantizeLinear", ["x", "scale", "zero_pt"], ["x_q"])
dq_node = helper.make_node("DequantizeLinear", ["x_q", "scale", "zero_pt"], ["x_dq"])
relu_node = helper.make_node("Relu", ["x_dq"], ["y"])

# Valid: Q -> DQ -> Relu
graph_good = helper.make_graph(
    [q_node, dq_node, relu_node], "quant_good",
    [X], [Y], initializer=[scale, zero_pt],
)
model_good = helper.make_model(graph_good, opset_imports=[helper.make_opsetid("", 17)])

v1 = check_quantization_contract(model_good)
print(f"  Valid Q->DQ pattern:   {'PASS' if not v1 else 'FAIL: ' + str(v1)}")

# Invalid: Q -> Relu (no DQ)
relu_bad = helper.make_node("Relu", ["x_q"], ["y"])  # feeds directly from Q
graph_bad = helper.make_graph(
    [q_node, relu_bad], "quant_bad",
    [X], [Y], initializer=[scale, zero_pt],
)
model_bad = helper.make_model(graph_bad, opset_imports=[helper.make_opsetid("", 17)])

v2 = check_quantization_contract(model_bad)
print(f"  Missing DQ pattern:    {'PASS' if not v2 else 'FAIL'}")
if v2:
    print(f"    → {v2[0]}")

<a id='section-6'></a>
## Section 6: Conformance Testing

### ONNX Backend Test Suite

The official ONNX project provides a **backend test suite** with per-operator test
cases. Each test case specifies:
- Input tensors
- Expected output tensors
- The ONNX model (single-node graph)

Runtimes (ORT, TVM, etc.) validate themselves against this suite.

### Numerical Accuracy Validation

For model-level conformance, we measure multiple error metrics:

**Absolute + Relative tolerance (allclose criterion):**

$$|y_{\text{ref}} - y_{\text{test}}| \leq \text{atol} + \text{rtol} \times |y_{\text{ref}}|$$

**Cosine similarity** for directional agreement:

$$\cos(\theta) = \frac{\mathbf{y}_{\text{ref}} \cdot \mathbf{y}_{\text{test}}}{\|\mathbf{y}_{\text{ref}}\| \; \|\mathbf{y}_{\text{test}}\|}$$

**Signal-to-Noise Ratio** (treats difference as noise):

$$\text{SNR} = 10 \log_{10} \frac{\|\mathbf{y}_{\text{ref}}\|^2}{\|\mathbf{y}_{\text{ref}} - \mathbf{y}_{\text{test}}\|^2} \;\text{dB}$$

Typical thresholds: SNR $> 30$ dB for float32, $> 20$ dB for float16.

### Cross-Runtime Consistency

```
                    Cross-Runtime Consistency Check
              ┌────────────────────────────────────────┐
              │                                        │
              │   Same ONNX Model + Same Input         │
              │        │           │          │        │
              │        ▼           ▼          ▼        │
              │   ┌────────┐ ┌────────┐ ┌────────┐   │
              │   │  ORT   │ │  ORT   │ │  ORT   │   │
              │   │  CPU   │ │  CUDA  │ │ TensorRT│  │
              │   └───┬────┘ └───┬────┘ └───┬────┘   │
              │       │          │          │         │
              │       ▼          ▼          ▼         │
              │     y_cpu     y_cuda     y_trt        │
              │       │          │          │         │
              │       └─── allclose? ───────┘         │
              │                                        │
              └────────────────────────────────────────┘
```

In [ ]:
# --- Numerical accuracy validation framework ---

def compute_error_metrics(y_ref, y_test):
    """Comprehensive numerical comparison between reference and test outputs."""
    y_ref = y_ref.flatten().astype(np.float64)
    y_test = y_test.flatten().astype(np.float64)

    abs_diff = np.abs(y_ref - y_test)

    norm_ref = np.linalg.norm(y_ref)
    norm_test = np.linalg.norm(y_test)
    cos_sim = np.dot(y_ref, y_test) / (norm_ref * norm_test + 1e-12)

    signal_power = np.sum(y_ref ** 2)
    noise_power = np.sum((y_ref - y_test) ** 2)
    snr_db = 10 * np.log10(signal_power / (noise_power + 1e-30))

    return {
        "max_abs_error": np.max(abs_diff),
        "mean_abs_error": np.mean(abs_diff),
        "cosine_similarity": cos_sim,
        "snr_db": snr_db,
    }


def conformance_test(model_proto, reference_fn, input_shapes,
                     n_samples=50, atol=1e-5, rtol=1e-4):
    """Run conformance tests: compare ORT output to reference."""
    sess = ort.InferenceSession(model_proto.SerializeToString())
    input_name = sess.get_inputs()[0].name

    results = []
    for i in range(n_samples):
        x = np.random.randn(*input_shapes).astype(np.float32)
        y_ref = reference_fn(x)
        y_ort = sess.run(None, {input_name: x})[0]

        metrics = compute_error_metrics(y_ref, y_ort)
        metrics["allclose"] = np.allclose(y_ref, y_ort, atol=atol, rtol=rtol)
        results.append(metrics)

    pass_rate = sum(r["allclose"] for r in results) / len(results)
    avg_snr = np.mean([r["snr_db"] for r in results])
    avg_cos = np.mean([r["cosine_similarity"] for r in results])
    max_err = max(r["max_abs_error"] for r in results)

    return {
        "pass_rate": pass_rate,
        "avg_snr_db": avg_snr,
        "avg_cosine_sim": avg_cos,
        "worst_max_abs_error": max_err,
        "n_samples": n_samples,
    }


# Run conformance test on our model
conf = conformance_test(model, ref_fn, (8, 784), n_samples=100)

print("Conformance Test Report")
print("=" * 50)
print(f"  Samples tested:       {conf['n_samples']}")
print(f"  Pass rate:            {conf['pass_rate']*100:.1f}%")
print(f"  Average SNR:          {conf['avg_snr_db']:.1f} dB")
print(f"  Average cosine sim:   {conf['avg_cosine_sim']:.10f}")
print(f"  Worst max abs error:  {conf['worst_max_abs_error']:.2e}")
print(f"\n  Verdict: {'CONFORMANT' if conf['pass_rate'] == 1.0 else 'NON-CONFORMANT'}")

In [ ]:
# --- Cross-runtime consistency (CPU with different optimization levels) ---

print("Cross-Configuration Consistency Check")
print("=" * 60)

configs = [
    ("CPU (no opt)", ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ("CPU (basic opt)", ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ("CPU (extended opt)", ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ("CPU (all opt)", ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

x_test = np.random.randn(8, 784).astype(np.float32)
model_bytes = model.SerializeToString()

outputs = {}
for name, opt_level in configs:
    so = ort.SessionOptions()
    so.graph_optimization_level = opt_level
    sess = ort.InferenceSession(model_bytes, so,
                                providers=["CPUExecutionProvider"])
    outputs[name] = sess.run(None, {"X": x_test})[0]

baseline_name = configs[0][0]
baseline_out = outputs[baseline_name]

print(f"\nBaseline: {baseline_name}")
print(f"{'Configuration':<22s} | {'Max Diff':>10s} | {'Cosine Sim':>14s} | {'Match':>5s}")
print("-" * 62)
for name, _ in configs[1:]:
    diff = np.max(np.abs(baseline_out - outputs[name]))
    m = compute_error_metrics(baseline_out, outputs[name])
    match = np.allclose(baseline_out, outputs[name], atol=1e-6)
    print(f"{name:<22s} | {diff:10.2e} | {m['cosine_similarity']:14.10f} | {'YES' if match else 'NO':>5s}")

<a id='section-7'></a>
## Section 7: Repair Strategies

When validation fails, we need strategies to **fix** models programmatically
rather than re-exporting from the source framework.

### Repair Taxonomy

```
                         Repair Strategies
                    ┌──────────┴──────────┐
                    │                     │
              Structural              Semantic
           ┌──────┼──────┐       ┌──────┼──────┐
           │      │      │       │      │      │
        Add    Rename  Remove  Convert  Add    Fix
       Missing  Name   Unused  Opset   Shape  Attribute
       Inits  Conflicts Nodes  Version Annot. Values
```

### Key Repair Operations

| Problem | Repair | API |
|:---|:---|:---|
| Missing initializer | Create from default values | `numpy_helper.from_array` |
| Name conflict | Rename with suffix | Walk `graph.node` and rename |
| Wrong opset version | Convert model opset | `onnx.version_converter` |
| Missing shape info | Run shape inference | `shape_inference.infer_shapes` |
| Unsupported op | Replace with equivalent subgraph | Manual node replacement |

In [ ]:
# --- Repair Strategy 1: Adding missing initializers ---

print("Repair 1: Adding Missing Initializers")
print("=" * 60)

def find_missing_initializers(model):
    """Find node inputs that are not graph inputs or initializers."""
    defined = set()
    for inp in model.graph.input:
        defined.add(inp.name)
    for init in model.graph.initializer:
        defined.add(init.name)
    for node in model.graph.node:
        for out in node.output:
            defined.add(out)

    missing = set()
    for node in model.graph.node:
        for inp in node.input:
            if inp and inp not in defined:
                missing.add(inp)
    return missing


def repair_missing_initializers(model, fill_value=0.0, fill_shape=(1,),
                                fill_dtype=np.float32):
    """Add zero-filled initializers for all missing inputs."""
    import copy
    model = copy.deepcopy(model)
    missing = find_missing_initializers(model)
    for name in missing:
        arr = np.full(fill_shape, fill_value, dtype=fill_dtype)
        init = numpy_helper.from_array(arr, name=name)
        model.graph.initializer.append(init)
    return model, missing


# Build a model with a missing initializer
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
add_node = helper.make_node("Add", ["X", "bias"], ["Y"])  # 'bias' not provided
graph = helper.make_graph([add_node], "missing_init", [X], [Y])
broken_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

missing = find_missing_initializers(broken_model)
print(f"  Missing inputs: {missing}")
check_and_report(broken_model, "Before repair")

repaired, added = repair_missing_initializers(
    broken_model, fill_value=0.0, fill_shape=(10,)
)
print(f"  Added initializers: {added}")
check_and_report(repaired, "After repair")

In [ ]:
# --- Repair Strategy 2: Opset version conversion ---

print("Repair 2: Opset Version Conversion")
print("=" * 60)

from onnx import version_converter

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
relu = helper.make_node("Relu", ["X"], ["Y"])
graph = helper.make_graph([relu], "opset_convert", [X], [Y])
model_v13 = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])

original_opset = model_v13.opset_import[0].version
print(f"  Original opset: {original_opset}")

target_opset = 17
model_v17 = version_converter.convert_version(model_v13, target_opset)
converted_opset = model_v17.opset_import[0].version
print(f"  Converted opset: {converted_opset}")

check_and_report(model_v17, f"After conversion to opset {target_opset}")

# Verify numerical equivalence after conversion
sess_v13 = ort.InferenceSession(model_v13.SerializeToString())
sess_v17 = ort.InferenceSession(model_v17.SerializeToString())

x_test = np.random.randn(4, 10).astype(np.float32)
y_v13 = sess_v13.run(None, {"X": x_test})[0]
y_v17 = sess_v17.run(None, {"X": x_test})[0]

max_diff = np.max(np.abs(y_v13 - y_v17))
print(f"  Numerical diff after opset conversion: {max_diff:.2e}")
print(f"  Exact match: {np.array_equal(y_v13, y_v17)}")

In [ ]:
# --- Repair Strategy 3: Shape annotation for incomplete models ---

print("Repair 3: Shape Annotation")
print("=" * 60)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)  # no shape

W_np = np.random.randn(784, 256).astype(np.float32)
B_np = np.random.randn(256).astype(np.float32)
W = numpy_helper.from_array(W_np, name="W")
B = numpy_helper.from_array(B_np, name="B")

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["X", "W"], ["H"]),
        helper.make_node("Add", ["H", "B"], ["HB"]),
        helper.make_node("Relu", ["HB"], ["Y"]),
    ],
    "shape_repair", [X], [Y], initializer=[W, B],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print("Before shape inference:")
print(f"  Intermediate value_info count: {len(model.graph.value_info)}")
out_type = model.graph.output[0].type.tensor_type
has_shape = out_type.HasField("shape")
print(f"  Output 'Y' has shape annotation: {has_shape}")

inferred = shape_inference.infer_shapes(model, check_type=True)

print("\nAfter shape inference:")
print(f"  Intermediate value_info count: {len(inferred.graph.value_info)}")
for vi in inferred.graph.value_info:
    tt = vi.type.tensor_type
    dtype = TensorProto.DataType.Name(tt.elem_type)
    dims = [d.dim_value for d in tt.shape.dim]
    print(f"    {vi.name}: {dtype}{dims}")

out_type = inferred.graph.output[0].type.tensor_type
dims = [d.dim_value for d in out_type.shape.dim]
dtype = TensorProto.DataType.Name(out_type.elem_type)
print(f"  Output 'Y': {dtype}{dims}")

In [ ]:
# --- Repair Strategy 4: Fixing name conflicts ---

print("Repair 4: Fixing Name Conflicts")
print("=" * 60)

def fix_name_conflicts(model, prefix="_fixed"):
    """Resolve SSA violations by appending unique suffixes to duplicate names."""
    import copy
    model = copy.deepcopy(model)

    seen = set()
    for inp in model.graph.input:
        seen.add(inp.name)
    for init in model.graph.initializer:
        seen.add(init.name)

    rename_map = {}
    counter = 0

    for node in model.graph.node:
        # Rename inputs according to existing rename map
        for i, inp in enumerate(node.input):
            if inp in rename_map:
                node.input[i] = rename_map[inp]

        # Check outputs for conflicts
        for i, out in enumerate(node.output):
            if out in seen:
                new_name = f"{out}{prefix}_{counter}"
                counter += 1
                rename_map[out] = new_name
                node.output[i] = new_name
            else:
                seen.add(out)

    # Update graph outputs
    for out in model.graph.output:
        if out.name in rename_map:
            out.name = rename_map[out.name]

    return model, rename_map


# Build a model with name conflicts
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
n1 = helper.make_node("Relu", ["X"], ["T"])
n2 = helper.make_node("Sigmoid", ["T"], ["T"])  # SSA violation: reuses 'T'
n3 = helper.make_node("Tanh", ["T"], ["Y"])
graph = helper.make_graph([n1, n2, n3], "name_conflict", [X], [Y])
bad_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

check_and_report(bad_model, "Before fix (SSA violation)")

fixed_model, renames = fix_name_conflicts(bad_model)
print(f"  Renames applied: {renames}")
check_and_report(fixed_model, "After fix (names deduplicated)")

print("\n  Fixed graph nodes:")
for node in fixed_model.graph.node:
    print(f"    {node.op_type}({', '.join(node.input)}) -> {list(node.output)}")

In [ ]:
# --- CI-grade comprehensive validation function ---

def full_ci_validation(model_proto, reference_fn=None, test_inputs=None,
                       policies=None, atol=1e-5, rtol=1e-4,
                       auto_repair=False):
    """
    Comprehensive CI validation combining structural, policy,
    and numerical checks with optional auto-repair.

    Returns a detailed report dict.
    """
    report = {
        "structural": None,
        "shape_inference": None,
        "policy": None,
        "runtime": None,
        "numerical": None,
        "repairs": [],
        "errors": [],
        "overall": None,
    }

    import copy
    model = copy.deepcopy(model_proto)

    # Stage 1: Structural
    try:
        checker.check_model(model)
        report["structural"] = "PASS"
    except Exception as e:
        if auto_repair:
            model, added = repair_missing_initializers(model, fill_shape=(1,))
            if added:
                report["repairs"].append(f"Added initializers: {added}")
            model, renames = fix_name_conflicts(model)
            if renames:
                report["repairs"].append(f"Renamed: {renames}")
            try:
                checker.check_model(model)
                report["structural"] = "PASS (after repair)"
            except Exception as e2:
                report["structural"] = "FAIL"
                report["errors"].append(str(e2)[:200])
                report["overall"] = "FAIL"
                return report
        else:
            report["structural"] = "FAIL"
            report["errors"].append(str(e)[:200])
            report["overall"] = "FAIL"
            return report

    # Stage 2: Shape inference
    try:
        inferred = shape_inference.infer_shapes(model, check_type=True)
        report["shape_inference"] = "PASS"
    except Exception as e:
        report["shape_inference"] = "FAIL"
        report["errors"].append(str(e)[:200])

    # Stage 3: Policy checks
    if policies:
        violations = run_all_policies(model, policies)
        report["policy"] = "FAIL" if violations else "PASS"
        if violations:
            for policy, vlist in violations.items():
                report["errors"].extend(vlist)
    else:
        report["policy"] = "SKIP"

    # Stage 4: Runtime
    try:
        sess = ort.InferenceSession(model.SerializeToString())
        report["runtime"] = "PASS"
    except Exception as e:
        report["runtime"] = "FAIL"
        report["errors"].append(str(e)[:200])
        report["overall"] = "FAIL"
        return report

    # Stage 5: Numerical
    if test_inputs is not None and reference_fn is not None:
        input_name = sess.get_inputs()[0].name
        max_diffs = []
        for x_np in test_inputs:
            y_ref = reference_fn(x_np)
            y_ort = sess.run(None, {input_name: x_np})[0]
            max_diffs.append(np.max(np.abs(y_ref - y_ort)))
        report["numerical"] = "PASS" if all(d < atol for d in max_diffs) else "FAIL"
        report["max_numerical_diff"] = max(max_diffs)
    else:
        report["numerical"] = "SKIP"

    failed = [k for k, v in report.items()
              if isinstance(v, str) and v == "FAIL"]
    report["overall"] = "FAIL" if failed else "PASS"

    return report


# Run on our valid model
report = full_ci_validation(
    valid_model,
    reference_fn=lambda x: x @ W_np + B_np,  # no relu for this model
    test_inputs=[np.random.randn(4, 784).astype(np.float32) for _ in range(10)],
)

print("Full CI Validation Report")
print("=" * 50)
for key in ["structural", "shape_inference", "policy", "runtime", "numerical", "overall"]:
    val = report[key]
    icon = "OK" if val and "PASS" in str(val) else "--" if val == "SKIP" else "XX"
    print(f"  [{icon}] {key:<20s}: {val}")
if report["repairs"]:
    print(f"  Repairs: {report['repairs']}")
if report.get("max_numerical_diff") is not None:
    print(f"  Max numerical diff: {report['max_numerical_diff']:.2e}")

In [ ]:
# --- Test auto-repair on a broken model and verify numerical equivalence ---

print("Auto-Repair + Numerical Equivalence Test")
print("=" * 60)

# Build a model that needs repair: missing initializer 'bias'
np.random.seed(123)
W_np_r = np.random.randn(10, 5).astype(np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W = numpy_helper.from_array(W_np_r, name="W")

mm = helper.make_node("MatMul", ["X", "W"], ["H"])
add = helper.make_node("Add", ["H", "bias"], ["Y"])  # 'bias' is missing
graph = helper.make_graph([mm, add], "needs_repair", [X], [Y], initializer=[W])
broken = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print("Step 1: Validate broken model")
check_and_report(broken, "Before repair")

print("\nStep 2: Auto-repair")
repaired, added = repair_missing_initializers(broken, fill_value=0.0, fill_shape=(5,))
print(f"  Added: {added}")
check_and_report(repaired, "After repair")

print("\nStep 3: Verify repaired model works at runtime")
sess = ort.InferenceSession(repaired.SerializeToString())
x_test = np.random.randn(4, 10).astype(np.float32)
y_ort = sess.run(None, {"X": x_test})[0]

# With bias=0, output should equal x @ W
y_expected = x_test @ W_np_r
max_diff = np.max(np.abs(y_ort - y_expected))
print(f"  Output shape: {y_ort.shape}")
print(f"  Max diff (vs x@W with zero bias): {max_diff:.2e}")
print(f"  Match: {np.allclose(y_ort, y_expected, atol=1e-5)}")

print("\nStep 4: Full CI validation of repaired model")
report = full_ci_validation(
    repaired,
    reference_fn=lambda x: x @ W_np_r,  # bias=0
    test_inputs=[np.random.randn(4, 10).astype(np.float32) for _ in range(20)],
)
print(f"  Overall: {report['overall']}")

In [ ]:
# --- Visualize the full validation pipeline results ---

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: validation stage pass rates across test scenarios
scenarios = ["Valid\nModel", "Missing\nInit", "Wrong\nOpset", "Shape\nError"]
stages = ["Structural", "Shape Inf.", "Runtime", "Numerical"]

pass_matrix = np.array([
    [1, 1, 1, 1],  # valid model
    [0, 0, 0, 0],  # missing init (fails stage 1)
    [0, 0, 0, 0],  # wrong opset (fails stage 1)
    [1, 1, 0, 0],  # shape error (fails stage 3)
])

colors_map = {1: '#4ECDC4', 0: '#FF6B6B'}
for i, scenario in enumerate(scenarios):
    for j, stage in enumerate(stages):
        color = colors_map[pass_matrix[i, j]]
        axes[0].barh(len(stages) - 1 - j + i * (len(stages) + 1),
                     1, color=color, edgecolor='white', linewidth=2)

ytick_pos = []
ytick_labels = []
for i, scenario in enumerate(scenarios):
    for j, stage in enumerate(stages):
        pos = len(stages) - 1 - j + i * (len(stages) + 1)
        ytick_pos.append(pos)
        ytick_labels.append(f"{stage}")

axes[0].set_yticks(ytick_pos)
axes[0].set_yticklabels(ytick_labels, fontsize=8)
axes[0].set_xlim(0, 1.5)
axes[0].set_xticks([])
axes[0].set_title("Validation Stage Results by Scenario", fontweight='bold')

for i, scenario in enumerate(scenarios):
    mid = (len(stages) - 1) / 2 + i * (len(stages) + 1)
    axes[0].text(1.1, mid, scenario, va='center', ha='left', fontsize=9,
                fontweight='bold')

pass_patch = mpatches.Patch(color='#4ECDC4', label='PASS')
fail_patch = mpatches.Patch(color='#FF6B6B', label='FAIL')
axes[0].legend(handles=[pass_patch, fail_patch], loc='lower right')

# Right: error distribution for conformance tests
np.random.seed(42)
y_ref = np.random.randn(1000).astype(np.float32)
noise_levels = [1e-7, 1e-5, 1e-3]
colors_noise = ['#45B7D1', '#FFA500', '#FF6B6B']

for sigma, color in zip(noise_levels, colors_noise):
    y_noisy = y_ref + np.random.randn(1000).astype(np.float32) * sigma
    errors = np.abs(y_ref - y_noisy)
    axes[1].hist(errors, bins=40, alpha=0.6, color=color,
                label=f'noise={sigma:.0e}', edgecolor='white')

axes[1].set_xlabel('Absolute Error', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Error Distribution by Noise Level', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_overview.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary

### Well-Formedness Predicate

$$\text{WF}(G) \iff \text{C1}(G) \wedge \text{C2}(G) \wedge \text{C3}(G) \wedge \text{C4}(G) \wedge \text{C5}(G)$$

| Condition | Property | Checker Mechanism |
|:---|:---|:---|
| C1 | $\forall n, \forall i \in \text{inputs}(n): i \in \text{defined\_before}(n)$ | Reference resolution |
| C2 | $\forall v_1 \neq v_2: \text{name}(v_1) \neq \text{name}(v_2)$ | SSA check |
| C3 | Node conforms to op schema for declared opset | Schema lookup |
| C4 | $\text{type}(v) \in \text{allowed\_types}(\text{op}(v))$ | Type inference |
| C5 | No cycles in the computation graph | Topological ordering |

### Validation Pipeline

| Stage | Tool | Cost |
|:---|:---|:---|
| Structural | `onnx.checker.check_model()` | ~1 ms |
| Shape/Type | `shape_inference.infer_shapes()` | ~10 ms |
| Runtime | `ort.InferenceSession()` | ~100 ms |
| Numerical | `np.allclose(y_ref, y_ort)` | ~seconds |
| Policy | Custom Python checks | ~1 ms |

### Key Repair Strategies

| Problem | Repair |
|:---|:---|
| Missing initializer | `numpy_helper.from_array(default, name)` |
| Name conflict | Rename with unique suffix |
| Wrong opset | `onnx.version_converter.convert_version()` |
| Missing shapes | `shape_inference.infer_shapes()` |

---

**Next:** [Model Validation — Apply Notebook](./Model_Validation_Apply.ipynb)